In [1]:
from langchain.chat_models import init_chat_model
from topic_gen.generate import Generator
from topic_gen.models import MTO_responds

from topic_gen import logger
logger.setLevel("INFO")

In [2]:
import ir_datasets

dataset = ir_datasets.load("disks45/nocr/trec-robust-2004")
store = dataset.docs_store()

queries_map = {}
for q in dataset.queries_iter():
    queries_map[q.query_id] = q
    
qrels_map = {}
for q in dataset.qrels_iter():
    if q.query_id not in qrels_map:
        qrels_map[q.query_id] = []
    qrels_map[q.query_id].append(q)

[INFO] Please confirm you agree to the TREC data usage agreement found at <https://trec.nist.gov/data/cd45/index.html>
[INFO] [starting] https://trec.nist.gov/data/robust/04.testset.gz
[INFO] [finished] https://trec.nist.gov/data/robust/04.testset.gz: [00:00] [34.3kB] [13.5MB/s]
[INFO] If you have a local copy of https://trec.nist.gov/data/robust/qrels.robust2004.txt, you can symlink it here to avoid downloading it again: /home/jueri/.ir_datasets/downloads/123c2a0ba2ec31178cb1050995dcfdfa
[INFO] [starting] https://trec.nist.gov/data/robust/qrels.robust2004.txt
[INFO] [finished] https://trec.nist.gov/data/robust/qrels.robust2004.txt: [00:00] [6.54MB] [18.8MB/s]


In [3]:
llm = init_chat_model(
    model="gemini-2.5-flash",
    model_provider="google_genai",
    temperature=0,
)

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

In [10]:
generator = Generator(llm=llm, output_class=MTO_responds)

In [11]:
query = queries_map["652"]

In [12]:
from ir_measures import Qrel

In [ ]:
my_qrels = []
for qrel in qrels_map[query.query_id][:10]:
    document = store.get(qrel.doc_id).default_text()
    res = generator.generate(
        prompt_name="robust-DNA-zero-shot", 
        document=document, 
        query=query.title, 
        narrative=query.narrative,
        description=query.description
        )
    my_qrels.append(Qrel(qrel.query_id, qrel.doc_id, res.O))


In [16]:
my_qrels

[Qrel(query_id='652', doc_id='FBIS3-13009', relevance=0, iteration='0'),
 Qrel(query_id='652', doc_id='FBIS3-13339', relevance=0, iteration='0'),
 Qrel(query_id='652', doc_id='FBIS3-13511', relevance=1, iteration='0'),
 Qrel(query_id='652', doc_id='FBIS3-13517', relevance=0, iteration='0'),
 Qrel(query_id='652', doc_id='FBIS3-13998', relevance=0, iteration='0'),
 Qrel(query_id='652', doc_id='FBIS3-15072', relevance=0, iteration='0'),
 Qrel(query_id='652', doc_id='FBIS3-15182', relevance=0, iteration='0'),
 Qrel(query_id='652', doc_id='FBIS3-15238', relevance=0, iteration='0'),
 Qrel(query_id='652', doc_id='FBIS3-15513', relevance=0, iteration='0'),
 Qrel(query_id='652', doc_id='FBIS3-15962', relevance=0, iteration='0')]